# E2a Proof-of-Principle - PARALLEL HEDGES (Colab Pro+)

Run this ALONGSIDE the main notebook (Pro+ allows multiple background sessions). Each parallel
copy trains a DIFFERENT variant toward the reviewer's "well below 1%" bar on oxide perovskites,
so if the baseline P1r plateaus above 1% you already have alternatives cooking.

**Set `VARIANT` (section 4) to a different value in each Colab notebook you open.**
Variants share the same Drive data + deterministically-rebuilt splits (same seeds as main notebook),
and each writes its OWN checkpoint postfix so they never collide. Self-contained end-to-end.

## 1. Mount Drive + paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT='/content/drive/MyDrive/chargeflow'
DATA_DIR=f'{DRIVE_ROOT}/data_perovskite'; OUTPUT_DIR=f'{DRIVE_ROOT}/output_perovskite'
LISTS_DIR=f'{DRIVE_ROOT}/lists_perovskite'
for d in (DRIVE_ROOT,DATA_DIR,OUTPUT_DIR,LISTS_DIR): os.makedirs(d,exist_ok=True)
print('Drive ready')

## 2. Clone repo + deps

In [ ]:
import os, shutil
REPO_DIR='/content/chargeflow'
if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone -q https://github.com/ngminhtri0394/chargeflow-electron-density-main.git {REPO_DIR}
for entry in os.listdir(REPO_DIR):
    if os.path.isdir(f'{REPO_DIR}/{entry}/scripts'): REPO_DIR=f'{REPO_DIR}/{entry}'; break
assert os.path.isdir(f'{REPO_DIR}/scripts'); print('repo:',REPO_DIR)

In [ ]:
%cd {REPO_DIR}
!pip -q install -e .
!pip -q install flow_matching ase pymatgen torchmetrics torchvision pyyaml
import torch; print('torch',torch.__version__,'cuda',torch.cuda.is_available())

## 3. Rebuild the oxide splits (deterministic - identical to main notebook)

In [ ]:
import re, glob, random, numpy as np
random.seed(42)
INPUT_ROOT=f'{DATA_DIR}/perovskites_rho_initial'; TARGET_ROOT=f'{DATA_DIR}/perovskites_rho'
def find_npy(root):
    r={}
    for sd in os.listdir(root):
        p=f'{root}/{sd}'
        if os.path.isdir(p):
            n=glob.glob(f'{p}/*.npy')
            if n: r[sd]=n[0]
    return r
inp=find_npy(INPUT_ROOT); tgt=find_npy(TARGET_ROOT)
common=sorted(set(inp)&set(tgt)); pairs=[(inp[k],tgt[k]) for k in common]
key2pair=dict(zip(common,pairs))
def parent_id_of(n):
    m=re.search(r'(mp-\d+)',n); return m.group(1) if m else n
def xsite_of(k):
    f=re.search(r'mp-\d+_([A-Za-z0-9]+)_remove',k)
    if not f: return '?'
    m=re.findall(r'([A-Z][a-z]?)\d*',f.group(1)); return m[-1] if m else '?'
def charge_of(k):
    m=re.search(r'_charge_(-?\d+)',k); return int(m.group(1)) if m else 0
def gridsize_path_for(npy):
    txt=npy.replace('.npy','.realshape.txt'); a=np.load(npy,mmap_mode='r')
    open(txt,'w').write(' '.join(map(str,a.shape))); return txt
def write_lists(sp,tag):
    dp=[p[0] for p in sp]; lp=[p[1] for p in sp]
    dg=[gridsize_path_for(x) for x in dp]; lg=[gridsize_path_for(x) for x in lp]
    for name,items in [(f'list_d_{tag}',dp),(f'list_l_{tag}',lp),(f'list_dgs_{tag}',dg),(f'list_lgs_{tag}',lg)]:
        open(f'{LISTS_DIR}/{name}','w').write('\n'.join(items)+'\n')
    print(f'  wrote {tag} ({len(sp)})')
def build(train_keys,test_keys,tag):
    write_lists([key2pair[k] for k in train_keys],f'{tag}_train')
    write_lists([key2pair[k] for k in test_keys], f'{tag}_test')
fam={}
for k in common: fam.setdefault(xsite_of(k),[]).append(k)
FAM_X=max(fam,key=lambda x:len(fam[x])); fk_all=fam[FAM_X]
fk=fk_all[:]; random.Random(1).shuffle(fk); nte=max(1,int(0.2*len(fk)))
build(fk[nte:],fk[:nte],f'perovskite_P1r_fam{FAM_X}')
chs=sorted({charge_of(k) for k in fk_all},key=abs); tc=chs[-1]
build([k for k in fk_all if charge_of(k)!=tc],[k for k in fk_all if charge_of(k)==tc],f'perovskite_P1_fam{FAM_X}')
P1R_TAG=f'perovskite_P1r_fam{FAM_X}'; P1_TAG=f'perovskite_P1_fam{FAM_X}'
print('family',FAM_X,'| P1R_TAG',P1R_TAG,'| P1_TAG',P1_TAG)

## 4. Pick a VARIANT (change this per notebook) + write config

| VARIANT | split | loss | core_weight | norm_rho | idea |
|---|---|---|---|---|---|
| `p1r_hybrid` | P1r | hybrid | 3 | off | optimize eps_MAE metric directly |
| `p1r_boost`  | P1r | l2     | 8 | on  | sharpen high-density + log-compress range |
| `p1r_combo`  | P1r | hybrid | 5 | on  | everything at once |
| `p1_hybrid`  | P1  | hybrid | 3 | off | charge-extrapolation + metric loss |

Run the main notebook's plain `p1r` (l2, cw=3) separately as the control.

In [ ]:
import yaml
VARIANT = 'p1r_hybrid'   # <-- CHANGE THIS in each parallel notebook

SPEC = {
  'p1r_hybrid': dict(tag=P1R_TAG, loss_type='hybrid', alpha=2.0, core_weight=3.0, norm_rho=False),
  'p1r_boost' : dict(tag=P1R_TAG, loss_type='l2',     alpha=2.0, core_weight=8.0, norm_rho=True),
  'p1r_combo' : dict(tag=P1R_TAG, loss_type='hybrid', alpha=2.0, core_weight=5.0, norm_rho=True),
  'p1_hybrid' : dict(tag=P1_TAG,  loss_type='hybrid', alpha=2.0, core_weight=3.0, norm_rho=False),
}[VARIANT]
EPOCHS=1500; POSTFIX=f'E2a_{VARIANT}'
cfg={
 'model':{'architecture':'unet_cond_xlarge','discrete_flow_matching':False,'use_ema':True},
 'data':{'train_data_list':f'{LISTS_DIR}/list_d_{SPEC["tag"]}_train',
         'train_label_list':f'{LISTS_DIR}/list_l_{SPEC["tag"]}_train',
         'train_data_gridsize':f'{LISTS_DIR}/list_dgs_{SPEC["tag"]}_train',
         'train_label_gridsize':f'{LISTS_DIR}/list_lgs_{SPEC["tag"]}_train',
         'use_charge_dataset':True,'data_augmentation':True,'downsample_data':1,'downsample_label':1,
         'normalize_density':SPEC['norm_rho'],'batch_size':1,'num_workers':2,'pin_memory':True},
 'training':{'epochs':EPOCHS,'start_epoch':0,'optimizer':'AdamW','learning_rate':1e-4,
         'optimizer_betas':[0.9,0.95],'decay_lr':True,'accum_iter':16,'class_drop_prob':0.1,
         'start_sad':True,'core_weight':SPEC['core_weight'],'loss_type':SPEC['loss_type'],
         'alpha':SPEC['alpha'],'save_frequency':50,'save_best':True,'log_frequency':10},
 'evaluation':{'eval_frequency':50,'compute_fid':False,'save_samples':True},
 'ode':{'method':'dopri5','options':{'atol':1e-5,'rtol':1e-5},'cfg_scale':1.0,
        'sampling_dtype':'float32','skewed_timesteps':True,'edm_schedule':True},
 'distributed':{'enabled':False,'backend':'nccl','init_method':'env://'},
 'output':{'output_dir':OUTPUT_DIR,'save_postfix':POSTFIX,'log_file':f'training_{POSTFIX}.log'},
 'seed':42,'device':'cuda','test_run':False,'eval_only':False}
CFG_PATH=f'{OUTPUT_DIR}/run_config_{POSTFIX}.yaml'
yaml.safe_dump(cfg,open(CFG_PATH,'w'),sort_keys=False)
print(f'VARIANT={VARIANT}  tag={SPEC["tag"]}  postfix={POSTFIX}')
print(f'  loss={SPEC["loss_type"]} alpha={SPEC["alpha"]} core_weight={SPEC["core_weight"]} norm_rho={SPEC["norm_rho"]}')

## 5. Train (resumable - background-execution safe)

In [ ]:
ckpt=f'{OUTPUT_DIR}/checkpoint-charged-residual-{POSTFIX}.pth'
resume_arg=f'--resume "{ckpt}"' if os.path.exists(ckpt) else ''
print('resume' if resume_arg else 'fresh','| training',VARIANT)
!cd {REPO_DIR} && python scripts/train.py --config "{CFG_PATH}" {resume_arg}

## 6. Eval this variant at high NFE (target: well below 1%)

In [ ]:
import sys, statistics, torch
for m in [k for k in list(sys.modules) if k=='src' or k.startswith('src.')]: del sys.modules[m]
sys.path.insert(0,REPO_DIR); os.chdir(REPO_DIR)
from src.data.dataset import RhoDatasetCharge
from src.models.model_configs import instantiate_model
from flow_matching.solver.ode_solver import ODESolver
from flow_matching.utils import ModelWrapper
from src.training.edm_time_discretization import get_time_discretization
class TrainArgs:
    def __init__(self,*a,**k): pass
device=torch.device('cuda'); ARCH='unet_cond_xlarge'; NFE=250
NORMRHO=SPEC['norm_rho']; TAG=SPEC['tag']
def scale_rho(r): return r.sign()*torch.log1p(1.25*r.abs())/4.2
def unscale_rho(s): return s.sign()*(torch.expm1(4.2*s.abs()))/1.25
ck=f'{OUTPUT_DIR}/best_model-{POSTFIX}.pth'
if not os.path.exists(ck): ck=f'{OUTPUT_DIR}/checkpoint-charged-residual-{POSTFIX}.pth'
assert os.path.exists(ck),'train first'
model=instantiate_model(architechture=ARCH,is_discrete=False,use_ema=True)
cp=torch.load(ck,map_location=device,weights_only=False); st=cp['model'] if 'model' in cp else cp
if list(st.keys())[0].startswith('module.'): st={k.replace('module.',''):v for k,v in st.items()}
model.load_state_dict(st,strict=True); model.to(device).eval()
class W(ModelWrapper):
    def forward(self,x,t,charge,concat):
        t=torch.zeros(x.shape[0],device=x.device)+t
        with torch.autocast('cuda'),torch.no_grad():
            return self.model(x,t,extra={'label':charge,'concat_conditioning':concat}).to(torch.float32)
solver=ODESolver(velocity_model=W(model).eval())
ds=RhoDatasetCharge(data_list_path=f'{LISTS_DIR}/list_d_{TAG}_test',label_list_path=f'{LISTS_DIR}/list_l_{TAG}_test',
    data_gridsize_path=f'{LISTS_DIR}/list_dgs_{TAG}_test',label_gridsize_path=f'{LISTS_DIR}/list_lgs_{TAG}_test',data_augmentation=False)
ld=torch.utils.data.DataLoader(ds,batch_size=1,shuffle=False); tg=get_time_discretization(nfes=NFE).to(device)
eps=[]
for i,(sad,target,charge) in enumerate(ld):
    sad=sad.to(device); target=target.to(device).float(); charge=charge.to(device)
    x0=scale_rho(sad) if NORMRHO else sad
    with torch.no_grad():
        out=solver.sample(time_grid=tg,x_init=x0,method='dopri5',atol=1e-6,rtol=1e-6,step_size=None,charge=charge,concat=x0).float()
    pred=unscale_rho(out) if NORMRHO else out
    eps.append(((pred-target).abs().sum()/target.sum()).item())
m=statistics.mean(eps)
print('='*56)
print(f'  VARIANT {VARIANT}  epoch {cp.get("epoch","?")}  n={len(eps)}  mean eps_MAE={m*100:.4f}%  median={statistics.median(eps)*100:.4f}%')
print(f'  frac<1%={sum(e<0.01 for e in eps)/len(eps)*100:.0f}%  frac<0.5%={sum(e<0.005 for e in eps)/len(eps)*100:.0f}%')
print(f'  reviewer bar (well below 1%): {"PASS" if m<0.01 else "not yet"}')
print('='*56)